# GC002 — T1-PM-KV 正式单 Seed 运行（前置门禁通过后）

本 Notebook 按顺序执行 T1-02、T1-03、T1-04 工程门禁；**只有三项全部 PASS，才会进入 T1-05 正式训练**。任何门禁失败都会保存并发布 INCOMPLETE/FAIL 报告，不启动正式训练。

正式配置已冻结：T1-PM-KV、Seed `173020021`、Warmup `97` 次更新、FP32、Batch 4、每 lane 64 个新预测目标、最多 64 个上下文复用 Token、250,000 个有效预测目标、977 次优化器更新。T1-04 的 50 Update / 12,800 Token 是单独的工程验证，不计入正式预算。

请在 Kaggle 开启 GPU（Tesla T4）、Internet，并启用已有 `GITHUB_TOKEN` 与 `KAGGLE_API_TOKEN` Secrets，然后选择 **Save & Run All**。无需挂载 Output；程序从已知版本恢复并逐文件校验数据和 Tokenizer。训练权重与检查点只保存在 Kaggle Output，不会提交到 Git。

本次只训练 T1-PM-KV 单 Seed。其他 GC002 轨道不获授权；GC001 历史结果不变；独立 Test 不访问；H1–H5 保持 `NOT_EVALUATED`。Notebook 不会在正式预算外自动启动下一轮训练。总运行硬上限两小时。


In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time

BRANCH = 'codex/gc002-t1-formal-readiness'
REPOSITORY = 'https://github.com/ArcheLabs/mini-cells.git'
WORKSPACE = Path('/kaggle/working')
REPO = WORKSPACE / 'minicells-gc002-t1-formal'
OUTPUT = WORKSPACE / 'epclm-gc002-t1-formal-bootstrap'
HARD_LIMIT_SECONDS = 7200
deadline = time.monotonic() + HARD_LIMIT_SECONDS

if not os.environ.get('KAGGLE_API_TOKEN', '').strip():
    from kaggle_secrets import UserSecretsClient
    os.environ['KAGGLE_API_TOKEN'] = UserSecretsClient().get_secret('KAGGLE_API_TOKEN').strip()
if not os.environ['KAGGLE_API_TOKEN']:
    raise RuntimeError('Enable the existing Kaggle Secret KAGGLE_API_TOKEN.')

def run_bounded(command, cwd=WORKSPACE, timeout_seconds=None):
    remaining = deadline - time.monotonic()
    if timeout_seconds is not None:
        remaining = min(remaining, timeout_seconds)
    if remaining <= 1:
        raise TimeoutError('The two-hour notebook budget is exhausted.')
    process = subprocess.Popen(command, cwd=cwd, text=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, start_new_session=(os.name == 'posix'))
    try:
        output, _ = process.communicate(timeout=remaining)
    except subprocess.TimeoutExpired:
        if os.name == 'posix': os.killpg(process.pid, signal.SIGTERM)
        else: process.terminate()
        try: output, _ = process.communicate(timeout=5)
        except subprocess.TimeoutExpired:
            if os.name == 'posix': os.killpg(process.pid, signal.SIGKILL)
            else: process.kill()
            output, _ = process.communicate()
        print(output or '')
        raise TimeoutError('Notebook time limit reached; recoverable output, if any, remains under ' + str(OUTPUT))
    print(output or '')
    if process.returncode != 0:
        raise RuntimeError('Command failed (' + str(process.returncode) + '). Output is under ' + str(OUTPUT))

if REPO.exists():
    if not (REPO / '.git').exists():
        import shutil
        shutil.rmtree(REPO)
if REPO.exists():
    origin = subprocess.run(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], capture_output=True, text=True)
    if origin.returncode != 0 or 'ArcheLabs/mini-cells' not in origin.stdout:
        raise RuntimeError(f'Refusing to use an unrelated directory: {REPO}')
    run_bounded(['git', '-C', str(REPO), 'fetch', '--depth=1', 'origin', BRANCH])
    run_bounded(['git', '-C', str(REPO), 'switch', '-C', BRANCH, 'FETCH_HEAD'])
    run_bounded(['git', '-C', str(REPO), 'clean', '-fd'])
else:
    run_bounded(['git', 'clone', '--depth=1', '--single-branch', '--branch', BRANCH,
        REPOSITORY, str(REPO)])

SOURCE_COMMIT = subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip()
OUTPUT = WORKSPACE / ('epclm-gc002-t1-formal-' + SOURCE_COMMIT[:12])

import torch
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU before running T1 formal validation/training.')
run_bounded([sys.executable, str(REPO / 'research/projects/cellular-language-models/experiments/epclm-core-001/gc002-fair-architecture/t1_formal.py'),
    '--output', str(OUTPUT)], cwd=REPO, timeout_seconds=6000)
